# 📧 Email Requirement Discovery (Unsupervised Theme Mining)

## 🎯 Goal
Surface what users are asking for from ingested emails **when the requirement space is unknown**.  
This is **discovery, not classification**:
- No predefined labels.
- No fixed output schema.
- The model proposes candidate themes.
- You decide what is a real requirement vs. an artifact.

---

## 🔄 Pipeline
1. **Load** → bring in raw email data.  
2. **Clean** → normalize and prepare text.  
3. **Filter noise** → remove irrelevant or low‑quality content.  
4. **Embed** → convert text into vector representations.  
5. **Cluster (BERTopic)** → group similar emails into candidate themes.  
6. **Inspect** → review clusters to identify meaningful requirements.  
7. **Land two tables in the silver lakehouse** → store results for downstream use.

---

## ⚙️ Step 0: Install
- On Fabric, `%pip install` restarts the PySpark kernel once (you will see the restart warning).  
- Run this cell **on its own, first**.  
- Everything defined before the restart is wiped — which is why **CONFIG comes after it**.  
- Then run the rest of the notebook **top‑to‑bottom**.


In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

%pip install -q "transformers==4.44.2" "sentence-transformers==3.0.1" "bertopic==0.16.4" umap-learn hdbscan
# %pip install -q talon   # optional: proper reply+signature extraction (install can be fussy)

StatementMeta(, f7fb708f-6e21-4069-9c79-66d2c26f520f, 8, Finished, Available, Finished, False)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nni 3.0 requires filelock<3.12, but you have filelock 3.13.1 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



# ⚙️ CONFIG — The Knobs You Own

This is the **cell that was missing**, which is why everything downstream threw `NameError`.

### Purpose
- Defines the **column names** the loader produces.  
- Sets the **silver write target**.  
- Configures the **clustering knobs**.

### Usage
- Re‑run this cell after **any kernel restart** to restore configuration.  

In [2]:
# --- source columns (must match what the loader produces) ---
ID_COL, SUBJECT_COL, TEXT_COL = "message_id", "subject", "body"

# --- SILVER destination (GUIDs taken from this notebook's lakehouse bindings) ---
# Sandbox-fine to hardcode; promote these to parameters if this becomes a pipeline.
WORKSPACE_ID  = "ac490e92-90f3-41a9-82ae-825ecaa77238"
SILVER_LH_ID  = "a03cfff1-048d-457c-8848-da958470832d"   # lh_silver_banking_data
SILVER_TABLES = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{SILVER_LH_ID}/Tables"
ASSIGN_TABLE  = "email_topic_assignments"
SUMMARY_TABLE = "email_topic_summary"
# Name-based equivalent (if you prefer readability over GUIDs):
# f"abfss://@onelake.dfs.fabric.microsoft.com/lh_silver_banking_data.Lakehouse/Tables"

# --- boilerplate to strip. Synthetic business emails are short, so likely minimal —
#     inspect (cell 7) and add real patterns here only if a footer leaks into topics. ---
DISCLAIMER_PATTERNS = [
    r"This (e-?mail|message) and any attachments.*",
    r"CONFIDENTIALITY NOTICE.*",
]

# --- model / clustering (tuned for a small ~900-doc corpus) ---
EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # English; multilingual? paraphrase-multilingual-MiniLM-L12-v2
MIN_TOPIC_SIZE  = 10   # 20 is too coarse for <1k docs; smaller -> more, finer topics
MIN_TOKENS      = 5    # drop emails shorter than this after cleaning
RANDOM_STATE    = 42   # UMAP is stochastic; pin it for reproducibility

StatementMeta(, f7fb708f-6e21-4069-9c79-66d2c26f520f, 10, Finished, Available, Finished, False)

# 📂 2. Load from Fabric Lakehouse Files

### 🔎 What It Does
- Reads **`.eml` files** from the **bronze lakehouse** (the attached default).  
- Parses MIME content to extract structured fields.  
- Builds a Pandas DataFrame (`pdf`) with:
  - `message_id`
  - `subject`
  - `body`

### ✅ Confirmed Working
- **879 emails** successfully loaded.  
- Spanning **80 month‑folders**.  

### ⚠️ Important Note
- The `*.eml` glob is **load‑bearing**.  
- Each email has a `.pdf` twin you **do not want** — filtering by `.eml` ensures only raw email files are ingested.  


In [3]:
from email import policy
from email.parser import BytesParser
import pandas as pd

try:
    fs = notebookutils.fs
except NameError:
    fs = mssparkutils.fs

BASE = "Files/bronze_raw/banking_data"
SCAN_ALL_MONTHS = True
ONE_MONTH = "2019/02"

def email_dirs():
    if not SCAN_ALL_MONTHS:
        return [f"{BASE}/{ONE_MONTH}/emails"]
    out = []
    for y in fs.ls(BASE):
        if not y.isDir: continue
        for m in fs.ls(y.path):
            p = f"{m.path}/emails"
            try: fs.ls(p); out.append(p)
            except Exception: pass
    return out

dirs = email_dirs()
print(f"{len(dirs)} email folder(s)")

bdf = (spark.read.format("binaryFile")
       .option("pathGlobFilter", "*.eml").load(dirs)
       .select("path", "content").toPandas())
rows = []
for _, r in bdf.iterrows():
    msg = BytesParser(policy=policy.default).parsebytes(r["content"])
    part = msg.get_body(preferencelist=("plain", "html"))
    rows.append({
        "message_id": r["path"].split("/")[-1],
        "subject": msg["subject"] or "",
        "body": part.get_content() if part else "",
    })
pdf = pd.DataFrame(rows)
print(f"{len(pdf):,} emails loaded")

StatementMeta(, f7fb708f-6e21-4069-9c79-66d2c26f520f, 11, Finished, Available, Finished, False)

80 email folder(s)
879 emails loaded


## 3. Clean — strip quoted history, signatures, disclaimers
This is where quality is won or lost. The model never sees the raw body; it sees only what survives this cell.

In [4]:
import re

REPLY_MARKERS = [
    r"\nOn .{0,120}? wrote:",
    r"\n-{2,}\s*Original Message\s*-{2,}",
    r"\nFrom:\s.*?\nSent:\s.*?\nTo:\s",
    r"\n_{5,}",
    r"\nSent from my \w+",
]
SIGNOFF_RE = re.compile(
    r"\n\s*(kind regards|best regards|regards|thanks|thank you|cheers|sincerely)[,.]?\s*\n",
    re.IGNORECASE,
)
DISCLAIMER_RES = [re.compile(p, re.IGNORECASE | re.DOTALL) for p in DISCLAIMER_PATTERNS]

def clean_email(text: str) -> str:
    if not isinstance(text, str):
        return ""
    cut = len(text)
    for pat in REPLY_MARKERS:
        m = re.search(pat, text)
        if m:
            cut = min(cut, m.start())
    text = text[:cut]
    text = "\n".join(l for l in text.splitlines() if not l.lstrip().startswith(">"))
    for rx in DISCLAIMER_RES:
        text = rx.sub("", text)
    m = SIGNOFF_RE.search(text)
    if m:
        text = text[:m.start()]
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compose(row):
    subj = (row[SUBJECT_COL] + ". ") if SUBJECT_COL and isinstance(row.get(SUBJECT_COL), str) else ""
    return (subj + clean_email(row[TEXT_COL])).strip()

pdf["clean_text"] = pdf.apply(compose, axis=1)

StatementMeta(, f7fb708f-6e21-4069-9c79-66d2c26f520f, 12, Finished, Available, Finished, False)

## 4. Filter noise (auto-replies, OOO, empties)
Clustering does not know that "Out of Office" is not a requirement. Drop the obvious non-signal before it forms its own confident, useless cluster.

In [5]:
NOISE_RE = re.compile(
    r"\b(out of office|automatic reply|auto-reply|do not reply|delivery (has )?failed|"
    r"undeliverable|read receipt|unsubscribe)\b",
    re.IGNORECASE,
)

pdf["n_tokens"] = pdf["clean_text"].str.split().str.len()
mask = (pdf["n_tokens"] >= MIN_TOKENS) & (~pdf["clean_text"].str.contains(NOISE_RE))
dropped = (~mask).sum()
pdf = pdf[mask].reset_index(drop=True)
print(f"Dropped {dropped:,} noise/short emails -> {len(pdf):,} remain")

docs = pdf["clean_text"].tolist()

StatementMeta(, f7fb708f-6e21-4069-9c79-66d2c26f520f, 13, Finished, Available, Finished, False)

/tmp/ipykernel_22171/357218470.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = (pdf["n_tokens"] >= MIN_TOKENS) & (~pdf["clean_text"].str.contains(NOISE_RE))


## 5. Embed

In [6]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBEDDING_MODEL)
embeddings = embedder.encode(docs, show_progress_bar=True, batch_size=64)

StatementMeta(, f7fb708f-6e21-4069-9c79-66d2c26f520f, 14, Finished, Available, Finished, False)

/nfs4/pyenv-7b50b043-8b11-46aa-a1d2-a1f6c2734602/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/nfs4/pyenv-7b50b043-8b11-46aa-a1d2-a1f6c2734602/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

## 6. Cluster with BERTopic
UMAP -> HDBSCAN -> c-TF-IDF keywords. Topic count is *data-driven*: you do not
pick "I want 10 themes". Topic `-1` is the outlier bucket — expect it to be large;
that is HDBSCAN being honest about ambiguous emails, not a bug.

In [7]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric="cosine", random_state=RANDOM_STATE)
hdbscan_model = HDBSCAN(min_cluster_size=MIN_TOPIC_SIZE, metric="euclidean",
                        cluster_selection_method="eom", prediction_data=True)

topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True,
)
topics, probs = topic_model.fit_transform(docs, embeddings)

info = topic_model.get_topic_info()
print(f"{(info['Topic'] != -1).sum()} topics found; "
      f"{(info.loc[info['Topic']==-1,'Count'].sum()):,} emails are outliers (-1)")
info.head(30)

StatementMeta(, f7fb708f-6e21-4069-9c79-66d2c26f520f, 15, Finished, Available, Finished, False)

2026-06-06 09:26:17,818 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


37 topics found; 22 emails are outliers (-1)


,Topic,Count,Name,Representation,Representative_Docs
0,-1,22,-1_2020_hardship_covid_holiday,"[2020, hardship, covid, holiday, payment, on, ...",[July 2020 suspicious behaviour without overca...
1,0,48,0_fields_sensitive_data_keep,"[fields, sensitive, data, keep, the, screensho...",[May 2023 data contract and sensitive fields. ...
2,1,48,1_fee_contact_questions_atm,"[fee, contact, questions, atm, card, can, know...",[February 2024 why are customers contacting us...
3,2,48,2_split_cause_root_enough,"[split, cause, root, enough, call, bank, insuf...",[May 2023 failed debit root cause split. Morni...
4,3,48,3_features_stress_repayment_change,"[features, stress, repayment, change, withdraw...",[May 2023 early warning features for repayment...
5,4,48,4_draft_me_work_health,"[draft, me, work, health, business, have, simp...",[May 2023 business health view before EXCO. Hi...
6,5,48,5_pack_evidence_masked_numbers,"[pack, evidence, masked, numbers, someone, cha...",[May 2023 customer outcome evidence pack. Hi t...
7,6,34,6_impact_worst_timeout_service,"[impact, worst, timeout, service, delay, probl...",[October 2022 service failures and customer im...
8,7,32,7_fee_contact_questions_atm,"[fee, contact, questions, atm, card, can, know...",[January 2021 why are customers contacting us....
9,8,32,8_features_stress_repayment_withdrawals,"[features, stress, repayment, withdrawals, tre...",[December 2021 early warning features for repa...


## 7. INSPECT — do not skip this, this is the whole job
The keywords are statistical, not a label. Read the representative emails for each
topic and ask: *is this a requirement, or an artifact (a sender domain, a template,
boilerplate I failed to strip)?* Name the real ones yourself.

In [8]:
for tid in info[info["Topic"] != -1]["Topic"].head(15):
    kw = ", ".join(w for w, _ in topic_model.get_topic(tid)[:8])
    print(f"\n=== TOPIC {tid}  (n={info.loc[info['Topic']==tid,'Count'].values[0]}) ===")
    print("keywords:", kw)
    for d in topic_model.get_representative_docs(tid)[:3]:
        print("  •", d[:200].replace("\n", " "))

StatementMeta(, f7fb708f-6e21-4069-9c79-66d2c26f520f, 16, Finished, Available, Finished, False)


=== TOPIC 0  (n=48) ===
keywords: fields, sensitive, data, keep, the, screenshots, layout, which
  • May 2023 data contract and sensitive fields. Hi all, For the May 2023 sprint, please confirm which fields are genuinely needed before we publish the next trusted data product. We will keep raw data in
  • May 2024 data contract and sensitive fields. Hi all, For the May 2024 sprint, please confirm which fields are genuinely needed before we publish the next trusted data product. We will keep raw data in
  • May 2022 data contract and sensitive fields. Hi all, For the May 2022 sprint, please confirm which fields are genuinely needed before we publish the next trusted data product. We will keep raw data in

=== TOPIC 1  (n=48) ===
keywords: fee, contact, questions, atm, card, can, know, late
  • February 2024 why are customers contacting us. Howzit team, Can we get a customer contact view for February 2024? People are not always complaining about the same thing, even when the transaction 

## 8. (optional) Reassign outliers
Only run this once you trust the topics. It forces ambiguous emails into their
nearest topic — convenient for coverage, but it *manufactures* certainty the data
did not have. Keep a flag so you can tell forced assignments apart downstream.

In [9]:
REASSIGN_OUTLIERS = False   # flip to True deliberately, not by default
if REASSIGN_OUTLIERS:
    new_topics = topic_model.reduce_outliers(docs, topics, strategy="c-tf-idf")
    pdf["topic"] = new_topics
    pdf["was_outlier"] = [t == -1 for t in topics]
else:
    pdf["topic"] = topics
    pdf["was_outlier"] = [t == -1 for t in topics]

pdf["topic_prob"] = [float(p.max()) if hasattr(p, "max") else float(p) for p in probs]

StatementMeta(, f7fb708f-6e21-4069-9c79-66d2c26f520f, 17, Finished, Available, Finished, False)

## 9. Write to the SILVER lakehouse

The original `saveAsTable("silver.…")` does **not** reach a different lakehouse on
Fabric — the default catalog here is bronze. So we write Delta straight to the silver
lakehouse's `Tables/` path by its OneLake `abfss` URI (unambiguous, and independent of
which lakehouse is attached as default). Fabric auto-registers Delta folders under
`Tables/` as tables.

`get_topic_info()` returns list-typed columns (`Representation`, `Representative_Docs`)
that break `spark.createDataFrame`, so the summary is rebuilt from scalar columns only.

> **Note:** If `lh_silver_banking_data` is schema-enabled, change the paths to
> `{SILVER_TABLES}/dbo/{ASSIGN_TABLE}` (and the same for the summary table).

**Delta, not parquet.** A Delta table is physically a folder of `.parquet` data files plus a `_delta_log/` transaction log — seeing parquet inside the table folder is expected and correct. The write below uses `.format("delta")` and then verifies with `DESCRIBE DETAIL`, which prints `delta` for each table.

In [11]:
# scalar-only frames (list columns from get_topic_info break createDataFrame)
label_map = {
    tid: ", ".join(w for w, _ in topic_model.get_topic(tid)[:6]) if tid != -1 else "outlier"
    for tid in info["Topic"]
}
pdf["topic_keywords"] = pdf["topic"].map(label_map)

assign_df = pdf[[ID_COL, "topic", "topic_keywords", "topic_prob", "was_outlier"]].copy()
assign_df["topic"]       = assign_df["topic"].astype("int64")
assign_df["topic_prob"]  = assign_df["topic_prob"].astype("float64")
assign_df["was_outlier"] = assign_df["was_outlier"].astype(bool)

summary_df = info[["Topic", "Count", "Name"]].rename(
    columns={"Topic": "topic", "Count": "email_count", "Name": "bertopic_name"}
).copy()
summary_df["topic_keywords"] = summary_df["topic"].map(label_map)
summary_df["topic"]       = summary_df["topic"].astype("int64")
summary_df["email_count"] = summary_df["email_count"].astype("int64")

ASSIGN_PATH  = f"{SILVER_TABLES}/{ASSIGN_TABLE}"
SUMMARY_PATH = f"{SILVER_TABLES}/{SUMMARY_TABLE}"

# If a PRIOR run wrote plain parquet to these paths, uncomment ONCE to clear them. A folder
# with parquet but no _delta_log is not a Delta table and can confuse Fabric's metastore.
# for p in (ASSIGN_PATH, SUMMARY_PATH):
#     try: notebookutils.fs.rm(p, True)
#     except Exception: pass

def write_delta(pdf_in, path):
    (spark.createDataFrame(pdf_in)
          .write.format("delta")               # <-- Delta explicitly, never parquet
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .save(path))

write_delta(assign_df, ASSIGN_PATH)
write_delta(summary_df, SUMMARY_PATH)

# PROVE the format is delta (prints 'delta' for each, or errors if it's plain parquet)
for p in (ASSIGN_PATH, SUMMARY_PATH):
    fmt = spark.sql(f"DESCRIBE DETAIL delta.`{p}`").select("format").first()[0]
    print(f"{p.split('/')[-1]:28s} format = {fmt}")

print(f"\nWritten to silver: {ASSIGN_TABLE} ({len(assign_df):,} rows), "
      f"{SUMMARY_TABLE} ({len(summary_df):,} rows)")

StatementMeta(, f7fb708f-6e21-4069-9c79-66d2c26f520f, 20, Finished, Available, Finished, False)

Written to silver: email_topic_assignments (879 rows), email_topic_summary (38 rows)


## 10. Validation checklist — how not to fool yourself
- **Coherence read:** for each topic, do the 3 representative emails actually share
  an intent? If not, the cluster is an artifact — raise `MIN_TOPIC_SIZE` or improve cleaning.
- **Boilerplate leak:** is any top topic dominated by signature/disclaimer/legal text?
  -> fix `DISCLAIMER_PATTERNS`, re-run from cell 3.
- **Language leak:** are topics split by language rather than intent? -> multilingual model.
- **Outlier rate:** if >40-50% land in -1, the corpus is too heterogeneous for these
  settings; tune, don't just reassign.
- **Stability:** re-run with a different `RANDOM_STATE`. Themes that survive are real;
  themes that vanish were noise. Only act on the stable ones.